# 資料を調べて答えるチャットボットを作る（複数ターンの RAG）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/25_dialogue/chat_rag.ipynb)

NLP クックブック 25-08 ／ 対話システム

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

対話の流れを踏まえながら、手元の資料を検索して答えるチャットボットを作りたい。雑談と資料に基づく回答を両立させたい。

In [1]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

docs = [
    "クックブック食堂の営業時間は平日 11 時から 15 時、土日は 11 時から 20 時です。月曜日は定休日です。",
    "クックブック食堂の看板メニューは、3 日間煮込んだ特製ビーフカレー (950 円) です。大盛りは 100 円増しです。",
    "クックブック食堂は 2019 年に京都市左京区で開業しました。店主の田中さんはインドで 2 年間修行しました。",
    "駐車場は店舗の裏に 3 台分あります。満車の場合は近隣のコインパーキングをご利用ください。",
    "予約は電話 (075-000-0000) のみ受け付けています。5 名以上の場合は前日までにご連絡ください。",
    "支払いは現金と QR コード決済に対応しています。クレジットカードは使えません。",
    "辛さは 5 段階から選べます。お子様向けの甘口カレー (600 円) もあります。",
]
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
doc_emb = embedder.encode(["passage: " + d for d in docs], normalize_embeddings=True)

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 3B の日本語モデル。T4 なら 1 応答あたり数秒〜10 秒ほど (GPU メモリ約 7GB)
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

def generate(messages, max_new_tokens=80):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [2]:
import re
from transformers import AutoModelForSequenceClassification, PreTrainedTokenizerFast

nli_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
# transformers 5 系の DebertaV2Tokenizer は全角の記号や英数字を [UNK] にしてしまう (issue #49622) ので、tokenizer.json から直接読む
nli_tok = PreTrainedTokenizerFast.from_pretrained(nli_name)
nli = AutoModelForSequenceClassification.from_pretrained(nli_name).eval()

def supported(context, reply):
    """応答の各文が資料から言えるか (含意確率の最小値)"""
    sents = [s for s in re.split(r"(?<=[。！？])", reply) if s.strip()]
    with torch.no_grad():
        probs = nli(**nli_tok([context] * len(sents), sents, return_tensors="pt", padding=True, truncation=True)).logits.softmax(-1)
    return probs[:, 0].min().item()

SYSTEM = "あなたはクックブック食堂の案内係です。丁寧な言葉で、1 文で短く答えてください。"
SMALL_TALK = re.compile(r"こんにちは|こんばんは|おはよう|はじめまして|ありがとう|助かりました|さようなら|また来ます")
REFERENCE = re.compile(r"それ|そこ|その|これ|あれ|そちら|ここ")
FALLBACK = "申し訳ありませんが、その件はわかりかねます。お店に直接お問い合わせください。"

class RagChat:
    def __init__(self):
        self.history = []

    def rewrite(self, question):
        """指示語を含む質問だけ、対話の文脈を補って検索用の質問にする"""
        if not self.history or not REFERENCE.search(question):
            return question
        dialogue = "\n".join(f"{'ユーザー' if m['role'] == 'user' else 'アシスタント'}: {m['content']}" for m in self.history[-4:])
        prompt = ("次の対話の最後の質問を、対話を読んでいない人にも意味が通じる、検索用の 1 つの質問に書き換えてください。"
                  f"書き換えた質問だけを出力してください。\n\n{dialogue}\nユーザー: {question}")
        return generate([{"role": "user", "content": prompt}], max_new_tokens=50).strip("「」")

    def say(self, question):
        if SMALL_TALK.search(question):                      # 挨拶やお礼は、検索せずにそのまま応答する
            reply, query, hits, score = generate([{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}], 40), None, [], None
        else:
            query = self.rewrite(question)
            sims = doc_emb @ embedder.encode("query: " + query, normalize_embeddings=True)
            hits = [int(i) for i in sims.argsort()[::-1][:2]]
            context = "\n".join(docs[i] for i in hits)
            # 資料は履歴に残さず、今回の質問にだけ付ける (履歴が資料で膨らむのを防ぐ)
            content = f"次の資料だけに基づいて質問に答えてください。\n\n# 資料\n{context}\n\n# 質問\n{query}"
            reply = generate([{"role": "system", "content": SYSTEM}, {"role": "user", "content": content}], 70)
            score = supported(context, reply)
            if score < 0.5:                                  # 資料から言えないことを述べていたら、定型の応答に差し替える
                reply = FALLBACK
        self.history += [{"role": "user", "content": question}, {"role": "assistant", "content": reply}]
        return reply, query, hits, score

bot = RagChat()
# 8 ターンの対話。T4 で 1 分弱
for user in ["こんにちは。", "おすすめのメニューは何ですか？", "それはいくらですか？", "子ども向けのメニューはありますか？",
             "日曜の 18 時は開いていますか？", "クレジットカードは使えますか？", "ペットを連れて行けますか？", "ありがとうございました。"]:
    reply, query, hits, score = bot.say(user)
    detail = "検索なし" if query is None else f"検索クエリ: {query} / 資料: {hits} / 裏付け: {score:.2f}"
    print(f"あなた: {user}\nボット: {reply}\n   ({detail})")

あなた: こんにちは。
ボット: こんにちは。本日はどのようなお料理をお探しでしょうか？当食堂では、様々なメニューをご用意しておりますので、お好みのお料理を見つけてください。何かお手伝いできることがあればお申し付けください。
   (検索なし)
あなた: おすすめのメニューは何ですか？
ボット: 申し訳ありませんが、その件はわかりかねます。お店に直接お問い合わせください。
   (検索クエリ: おすすめのメニューは何ですか？ / 資料: [1, 6] / 裏付け: 0.09)
あなた: それはいくらですか？
ボット: 申し訳ありませんが、その件はわかりかねます。お店に直接お問い合わせください。
   (検索クエリ: この食堂で特に人気のある料理やおすすめのメニューについて情報を得るには、どのように問い合わせるのが最適ですか？ / 資料: [4, 1] / 裏付け: 0.00)
あなた: 子ども向けのメニューはありますか？
ボット: はい、クックブック食堂ではお子様向けの甘口カレーをご用意しており、600円でお召し上がりいただけます。
   (検索クエリ: 子ども向けのメニューはありますか？ / 資料: [6, 1] / 裏付け: 1.00)
あなた: 日曜の 18 時は開いていますか？
ボット: はい、日曜の 18 時は開いております。クックブック食堂の土日の営業時間は 11 時から 20 時までです。
   (検索クエリ: 日曜の 18 時は開いていますか？ / 資料: [0, 4] / 裏付け: 0.71)
あなた: クレジットカードは使えますか？
ボット: 申し訳ありませんが、その件はわかりかねます。お店に直接お問い合わせください。
   (検索クエリ: クレジットカードは使えますか？ / 資料: [5, 3] / 裏付け: 0.00)
あなた: ペットを連れて行けますか？
ボット: 申し訳ありませんが、その件はわかりかねます。お店に直接お問い合わせください。
   (検索クエリ: ペットを連れて行けますか？ / 資料: [4, 5] / 裏付け: 0.00)
あなた: ありがとうございました。
ボット: どういたしまして、またのご利用をお待ちしております。
   (検索なし)


- 各ターンで「（必要なら）質問の書き換え → 検索 → 資料を付けて生成 → 資料との照合」を行う。
- 小さな埋め込みモデルの類似度では、挨拶と質問、答えのある質問とない質問をうまく区別できない（挨拶のほうが類似度が高く出ることもある）。そこで、挨拶は規則で判定し、答えが資料にあるかどうかは生成した応答を NLI で資料と照合して確かめている。
- 3B クラスのモデルは、資料にないこと（ペット）を聞かれると、もっともらしい答えを作ってしまうことが多い。照合で裏付けが取れない応答は定型文に差し替えると、誤った案内を防げる。そのかわり、答えが資料にある質問まで差し替えてしまうこともある。この例では、2 ターン目のおすすめメニュー（裏付け 0.09）と 6 ターン目のクレジットカード (0.00) が、正しい資料を検索できているのに定型文になった。安全側に倒す設計で、しきい値や NLI のモデルを変えると結果が変わる。
- 差し替えた定型文は履歴にも残るので、次のターンの指示語が指す先がなくなる。3 ターン目の「それはいくらですか？」は、直前の応答が定型文だったため、「この食堂で特に人気のある料理やおすすめのメニューについて情報を得るには、どのように問い合わせるのが最適ですか？」という別の質問に書き換えられた。値段の資料は 2 番目に検索できていたが、質問がずれているので応答の裏付けは 0.00 になり、定型文に差し替えられた。
- 質問の書き換えは、指示語を含むときだけ行う。毎回書き換えると、関係のない前の話題を持ち込んで検索を外すことがある（RAG の章の「検索しやすい形に質問を書き換える（文脈の補完と HyDE）」を参照）。
- 検索した資料は、そのターンの入力にだけ付けて、履歴には質問と回答だけを残す。